<a href="https://colab.research.google.com/github/ASHIKAJAN/ABC-staff-company/blob/main/AI_powered_paraphrasing_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers sentencepiece sentence-transformers language-tool-python rouge-score nltk

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu

In [ ]:
print("Loading Models...")

tokenizer = AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws")
model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws")

grammar_tool = language_tool_python.LanguageTool('en-US')

similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Models Loaded Successfully!")


In [ ]:
def paraphrase(text):

    sentence = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        sentence,
        max_length=256,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    outputs = model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_length=256,
        do_sample=True,
        top_k=120,
        top_p=0.95,
        early_stopping=True,
        num_return_sequences=1
    )

    paraphrased = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return paraphrased

In [ ]:
def grammar_correct(text):

    matches = grammar_tool.check(text)
    corrected = language_tool_python.utils.correct(text, matches)

    return corrected


In [ ]:
def evaluate(original, paraphrased):

    print("\nEvaluation Metrics")

    # BLEU Score
    bleu = sentence_bleu(
        [original.split()],
        paraphrased.split()
    )

    # ROUGE
    scorer = rouge_scorer.RougeScorer(
        ['rouge1','rouge2','rougeL'],
        use_stemmer=True
    )

    rouge = scorer.score(original, paraphrased)

    # Semantic Similarity
    emb1 = similarity_model.encode(original, convert_to_tensor=True)
    emb2 = similarity_model.encode(paraphrased, convert_to_tensor=True)

    similarity = util.cos_sim(emb1, emb2).item()

    print("--------------------------------")
    print("BLEU Score :", round(bleu,4))
    print("ROUGE-1 :", round(rouge['rouge1'].fmeasure,4))
    print("ROUGE-2 :", round(rouge['rouge2'].fmeasure,4))
    print("ROUGE-L :", round(rouge['rougeL'].fmeasure,4))
    print("Semantic Similarity :", round(similarity,4))
    print("--------------------------------")


In [ ]:
print("\n========== AI PARAPHRASING TOOL ==========\n")

text = input("Artificial Intelligence is transforming the healthcare industry by improving diagnosis, treatment planning, and patient care.:\n")

print("\nGenerating Paraphrase...\n")

paraphrased = paraphrase(text)

corrected = grammar_correct(paraphrased)

print("Original Text\n")
print(text)

print("\nParaphrased Text\n")
print(paraphrased)

print("\nGrammar Corrected Output\n")
print(corrected)

evaluate(text, corrected)